# S0 — CMIP6 SSP Scenario Catalog Compatibility / CMIP6 SSP情景目录兼容性分析

This notebook checks the availability of SSP scenario data for models and variables already selected by the historical pipeline (caseA/S0 + S0.1).

**Scenarios:** SSP126 (low), SSP245 (medium), SSP585 (high).

**Approach:** The historical S0/S0.1 pipeline already selected 35 family representatives, their best members, and 18 target variables. Here we:

1. Load the historical model/member/variable selections
2. For each SSP scenario, query the **same local ESGF catalog** (if an SSP-specific catalog exists) or note that S1 will verify availability via live ESGF queries
3. Fixed fields (sftlf, areacella, sftgif) are scenario-independent — reuse from historical

**中文说明：** 本Notebook检查历史分析管线（caseA）已选模型/变量在SSP情景下的数据可用性。不重新选模型——直接复用historical S0.1的选择，检查哪些在SSP126/245/585下可用。固定场与情景无关，直接复用历史下载。

## 1. Load historical selections / 加载历史管线选择结果

Load the model/member/variable selections from the historical pipeline (caseA/S0.1).
No SSP-specific ESGF catalog exists locally, so actual SSP data availability will be verified in S1 via live ESGF queries.

**中文说明：** 从历史管线（caseA/S0.1）加载已选模型、member和变量。本地没有SSP的ESGF目录，实际SSP数据可用性在S1中通过ESGF在线查询确认。

In [1]:
from __future__ import annotations

from pathlib import Path

import pandas as pd
from IPython.display import display


def locate_case_dir() -> Path:
    for d in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if d.name == "caseA":
            return d
        nested = d / "case" / "caseA"
        if nested.is_dir():
            return nested
    raise FileNotFoundError("Could not locate case/caseA")


CASE_DIR = locate_case_dir()
S01_HIST_DIR = CASE_DIR / "output" / "S0.1"

FUTURE_DIR = Path.cwd().resolve()
if FUTURE_DIR.name != "case_future2609":
    for d in [FUTURE_DIR, *FUTURE_DIR.parents]:
        nested = d / "case" / "case_future2609"
        if nested.is_dir():
            FUTURE_DIR = nested
            break

OUT_DIR = FUTURE_DIR / "output" / "S0"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SCENARIOS = ["ssp126", "ssp245", "ssp585"]
SSP_ACTIVITY_ID = "ScenarioMIP"
SSP_START_YEAR = 2015
SSP_END_YEAR = 2100

print("Historical caseA:", CASE_DIR)
print("Historical S0.1:", S01_HIST_DIR, "exists=", S01_HIST_DIR.is_dir())
print("Future case dir:", FUTURE_DIR)
print("Output:", OUT_DIR)
print(f"Scenarios: {SCENARIOS}")
print(f"Period: {SSP_START_YEAR}–{SSP_END_YEAR}")

Historical caseA: /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/case/caseA
Historical S0.1: /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/case/caseA/output/S0.1 exists= True
Future case dir: /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/case/case_future2609
Output: /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/case/case_future2609/output/S0
Scenarios: ['ssp126', 'ssp245', 'ssp585']
Period: 2015–2100


In [2]:
selected_models = pd.read_csv(S01_HIST_DIR / "S0.1_selected_models.csv")
download_manifest = pd.read_csv(S01_HIST_DIR / "S0.1_download_available.csv")

print(f"Historical S0.1 selected models: {len(selected_models)}")
print(f"Historical S0.1 downloadable records: {len(download_manifest)}")
print()
print("=== Selected models & members ===")
display(selected_models[["family", "model", "member_id", "pool",
                          "n_target_available", "n_target_missing", "missing"]])

Historical S0.1 selected models: 35
Historical S0.1 downloadable records: 629

=== Selected models & members ===


,family,model,member_id,pool,n_target_available,n_target_missing,missing
0,CESM2,CESM2,r1i1p1f1,core,18,0,NaN
1,CNRM,CNRM-CM6-1,r1i1p1f2,core,18,0,NaN
2,CanESM,CanESM5,r1i1p1f1,core,18,0,NaN
3,E3SM,E3SM-1-0,r1i1p1f1,core,18,0,NaN
4,GFDL,GFDL-CM4,r1i1p1f1,core,18,0,NaN
5,IPSL-CM6A,IPSL-CM6A-LR,r1i1p1f1,core,18,0,NaN
6,MIROC-ES2,MIROC-ES2H,r1i1p4f2,core,18,0,NaN
7,MRI,MRI-ESM2-0,r1i2p1f1,core,18,0,NaN
8,SAM0,SAM0-UNICON,r1i1p1f1,core,18,0,NaN
9,TaiESM,TaiESM1,r1i1p1f1,core,18,0,NaN


## 2. SSP download plan / SSP下载计划

For each SSP scenario, the plan is to download the **same models, members, and variables** as historical.
The key differences from historical:
- **activity_id**: `ScenarioMIP` (not `CMIP`)
- **experiment_id**: `ssp126` / `ssp245` / `ssp585` (not `historical`)
- **Time period**: 2015–2100 (not 1985–2014)
- **Fixed fields**: reuse from historical (sftlf, areacella are scenario-independent)

Not all models may have all SSP scenarios available — S1 will verify via live ESGF queries and skip unavailable combinations.

**中文说明：** 每个SSP情景下载与历史相同的模型、member和变量。与历史的区别：activity_id改为ScenarioMIP，experiment_id改为ssp126/245/585，时间段改为2015–2100。固定场（sftlf/areacella）与情景无关，直接复用历史已下载的。不是所有模型都有所有SSP情景——S1会在线查询确认，跳过不可用的组合。

In [3]:
ssp_plan_rows = []
for scenario in SCENARIOS:
    for _, row in download_manifest.iterrows():
        ssp_plan_rows.append({
            "scenario": scenario,
            "activity_id": SSP_ACTIVITY_ID,
            "model": row["model"],
            "family": row["family"],
            "member_id": row["member_id"],
            "variable": row["variable"],
            "table_id": row["table_id"],
            "grid_label": row["grid_label"],
            "category": row["category"],
            "historical_status": row["status"],
            "ssp_status": "planned",
        })

ssp_plan = pd.DataFrame(ssp_plan_rows)

ssp_time_plan = ssp_plan[ssp_plan["category"] != "fixed"].copy()

print(f"=== SSP download plan ===")
print(f"  Scenarios: {SCENARIOS}")
print(f"  Period: {SSP_START_YEAR}–{SSP_END_YEAR}")
print(f"  Activity: {SSP_ACTIVITY_ID}")
print(f"  Models: {ssp_time_plan['model'].nunique()}")
print(f"  Total planned records (excl. fixed fields): {len(ssp_time_plan)}")
print(f"  Per scenario: {len(ssp_time_plan) // len(SCENARIOS)}")
print()

per_scenario = ssp_time_plan.groupby("scenario").agg(
    n_models=("model", "nunique"),
    n_records=("variable", "count"),
    n_core=("category", lambda x: (x == "core").sum()),
    n_context=("category", lambda x: (x == "context").sum()),
).reset_index()
print("Records per scenario:")
display(per_scenario)

print()
print("Fixed fields (reuse from historical — NOT re-downloaded):")
fixed_plan = ssp_plan[ssp_plan["category"] == "fixed"].drop_duplicates(
    subset=["model", "variable"]
)
print(f"  {len(fixed_plan)} fixed field records across {fixed_plan['model'].nunique()} models")

=== SSP download plan ===
  Scenarios: ['ssp126', 'ssp245', 'ssp585']
  Period: 2015–2100
  Activity: ScenarioMIP
  Models: 35
  Total planned records (excl. fixed fields): 1746
  Per scenario: 582

Records per scenario:


,scenario,n_models,n_records,n_core,n_context
0,ssp126,35,582,103,479
1,ssp245,35,582,103,479
2,ssp585,35,582,103,479



Fixed fields (reuse from historical — NOT re-downloaded):
  47 fixed field records across 25 models


In [4]:
ssp_time_plan.to_csv(OUT_DIR / "S0_ssp_download_plan.csv", index=False)
selected_models.to_csv(OUT_DIR / "S0_historical_selected_models.csv", index=False)

print("Saved to", OUT_DIR)
for f in sorted(OUT_DIR.glob("S0_*.csv")):
    print(" ", f.name)
print()
print("Next: S0.1 generates per-scenario download manifests, S1 queries ESGF and downloads.")

Saved to /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/case/case_future2609/output/S0
  S0_historical_selected_models.csv
  S0_ssp_download_plan.csv

Next: S0.1 generates per-scenario download manifests, S1 queries ESGF and downloads.
